# UrbanGap: H3 grid and toilet assignment

Inspect resolution-9 hexagons and the historical placement proxy. H3 selects cells by their centres; cells remain complete and can cross the SA1 study boundary. Inputs: interim study/grid and authoritative features. Outputs: checks and maps. Grid generation and assignment live in `src/urbangap/grid.py`.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from urbangap.common import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


In [ ]:
grid = gpd.read_file(INTERIM / "grid/sydney_hex_grid.geojson")
assert grid.crs.to_epsg() == 7856
assert len(grid) == 16228 and grid.hex_id.is_unique
assert grid.toilet_count.sum() == 2570
assert (grid.toilet_count > 0).sum() == 1911
display(grid.drop(columns="geometry").describe())

## Boundary semantics

National toilet coordinates are assigned directly with H3 resolution 9, then joined to every selected cell. Zero-count cells remain in the prediction grid. This differs intentionally from clipping toilet points to the SA1 union. Missing coordinates are excluded and reported by the pipeline.

In [ ]:
from urbangap.grid import build_grid
study = gpd.read_file(INTERIM / "boundaries/sydney_study_sa1_final.geojson")
rebuilt = build_grid(study)
assert set(rebuilt.hex_id) == set(grid.hex_id)
print("Regenerated H3 membership matches exactly.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
grid.plot(ax=ax, column="toilet_count", cmap="YlOrRd", legend=True, linewidth=0)
ax.set(title="Mapped toilet facilities per complete H3 cell")
ax.set_axis_off()
plt.show()